A study of perception vs reality in crime - South Yorkshire

In [42]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import json

In [43]:
perception_df = pd.read_csv('Perceptions_CJS_England and Wales_2024Q4.csv')

perception_df.head()

,Year,Quarter,MeasurementVar,MeasurementLevel,MeasurementType,Geography,HouseholdType,Sex,Age,CharacteristicVar,Characteristic,Estimate,StandardError,UnweightedCount
0,2024,4,catt2,Person,Perception,England and Wales,,All people,16-24,accharm3,Detached house,49.81129436,3.989754607,229
1,2024,4,catt2,Person,Perception,England and Wales,,All people,16-24,accharm3,Flats/maisonettes,45.58845912,3.317115821,460
2,2024,4,catt2,Person,Perception,England and Wales,,All people,16-24,accharm3,House,51.5229105,1.880560431,1194
3,2024,4,catt2,Person,Perception,England and Wales,,All people,16-24,accharm3,Other accommodation,[x],[x],[x]
4,2024,4,catt2,Person,Perception,England and Wales,,All people,16-24,accharm3,Semi-detached house,52.39973521,3.208098498,418


In [44]:
perception_df.columns

Index(['Year', 'Quarter', 'MeasurementVar', 'MeasurementLevel',
       'MeasurementType', 'Geography', 'HouseholdType', 'Sex', 'Age',
       'CharacteristicVar', 'Characteristic', 'Estimate', 'StandardError',
       'UnweightedCount'],
      dtype='object')

In [45]:
perception_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28509 entries, 0 to 28508
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Year               28509 non-null  int64 
 1   Quarter            28509 non-null  int64 
 2   MeasurementVar     28509 non-null  object
 3   MeasurementLevel   28509 non-null  object
 4   MeasurementType    28509 non-null  object
 5   Geography          28509 non-null  object
 6   HouseholdType      28509 non-null  object
 7   Sex                28509 non-null  object
 8   Age                28509 non-null  object
 9   CharacteristicVar  28509 non-null  object
 10  Characteristic     27573 non-null  object
 11  Estimate           28509 non-null  object
 12  StandardError      28509 non-null  object
 13  UnweightedCount    28509 non-null  object
dtypes: int64(2), object(12)
memory usage: 3.0+ MB


In [46]:
# Look at only rows where MeasurementType = Perception and Characteristic = Yorkshire and The Humber
perception_df = perception_df[(perception_df['MeasurementType'] == 'Perception') & (perception_df['Characteristic'] == 'Yorkshire and The Humber')]   
perception_df.head()


,Year,Quarter,MeasurementVar,MeasurementLevel,MeasurementType,Geography,HouseholdType,Sex,Age,CharacteristicVar,Characteristic,Estimate,StandardError,UnweightedCount
28,2024,4,catt2,Person,Perception,England and Wales,,All people,16-24,gor,Yorkshire and The Humber,51.84769986,5.633952562,170
126,2024,4,catt2,Person,Perception,England and Wales,,All people,16+,gor,Yorkshire and The Humber,42.40532267,1.341798482,3209
218,2024,4,catt2,Person,Perception,England and Wales,,All people,25-34,gor,Yorkshire and The Humber,43.83883414,3.071650051,439
309,2024,4,catt2,Person,Perception,England and Wales,,All people,35-44,gor,Yorkshire and The Humber,41.67928107,2.801424804,551
401,2024,4,catt2,Person,Perception,England and Wales,,All people,45-54,gor,Yorkshire and The Humber,34.92154708,2.796132458,447


In [47]:

# match value with the values listed in the refrence guide for MeasurementVar 'reference-measurementvar.csv'
# A key will need to be created to understand the numeric value and the measurement label from the refrence csv
measurement_var_df = pd.read_csv('reference-measurementvar.csv')
measurement_var_df.head()


,MeasurementVar,MeasurementLabel,MeasurementLevel,MeasurementType
0,abancar1,Perceived problem with abandoned or burnt out ...,Person,Perception
1,acquai_i,Acquaintance violence,Person,Incidence rate
2,acquai_p,Acquaintance violence,Person,Prevalence rate
3,afee_i,Advance fee fraud,Person,Incidence rate
4,afee_p,Advance fee fraud,Person,Prevalence rate


In [48]:
# Assign a numeric value to the measurement variable
#This will be used to create a key to understand the numeric value and the measurement label from the refrence csv
measurement_var_df['MeasurementVar'] = measurement_var_df['MeasurementVar'].astype('category')

In [49]:
measurement_var_df.head()

,MeasurementVar,MeasurementLabel,MeasurementLevel,MeasurementType
0,abancar1,Perceived problem with abandoned or burnt out ...,Person,Perception
1,acquai_i,Acquaintance violence,Person,Incidence rate
2,acquai_p,Acquaintance violence,Person,Prevalence rate
3,afee_i,Advance fee fraud,Person,Incidence rate
4,afee_p,Advance fee fraud,Person,Prevalence rate


In [50]:
# Create a numeric value column for the measurement variable without removing the original column
measurement_var_df['MeasurementVar_numeric'] = measurement_var_df['MeasurementVar'].cat.codes
measurement_var_df.head()

,MeasurementVar,MeasurementLabel,MeasurementLevel,MeasurementType,MeasurementVar_numeric
0,abancar1,Perceived problem with abandoned or burnt out ...,Person,Perception,0
1,acquai_i,Acquaintance violence,Person,Incidence rate,1
2,acquai_p,Acquaintance violence,Person,Prevalence rate,2
3,afee_i,Advance fee fraud,Person,Incidence rate,3
4,afee_p,Advance fee fraud,Person,Prevalence rate,4


In [51]:
print(measurement_var_df.columns)
print(perception_df.columns)

Index(['MeasurementVar', 'MeasurementLabel', 'MeasurementLevel',
       'MeasurementType', 'MeasurementVar_numeric'],
      dtype='object')
Index(['Year', 'Quarter', 'MeasurementVar', 'MeasurementLevel',
       'MeasurementType', 'Geography', 'HouseholdType', 'Sex', 'Age',
       'CharacteristicVar', 'Characteristic', 'Estimate', 'StandardError',
       'UnweightedCount'],
      dtype='object')


In [52]:
merge_df = perception_df.merge(
    measurement_var_df[['MeasurementVar', 'MeasurementVar_numeric']],
    on='MeasurementVar',
    how='left'
)


In [53]:
merge_df.head()

,Year,Quarter,MeasurementVar,MeasurementLevel,MeasurementType,Geography,HouseholdType,Sex,Age,CharacteristicVar,Characteristic,Estimate,StandardError,UnweightedCount,MeasurementVar_numeric
0,2024,4,catt2,Person,Perception,England and Wales,,All people,16-24,gor,Yorkshire and The Humber,51.84769986,5.633952562,170,50
1,2024,4,catt2,Person,Perception,England and Wales,,All people,16+,gor,Yorkshire and The Humber,42.40532267,1.341798482,3209,50
2,2024,4,catt2,Person,Perception,England and Wales,,All people,25-34,gor,Yorkshire and The Humber,43.83883414,3.071650051,439,50
3,2024,4,catt2,Person,Perception,England and Wales,,All people,35-44,gor,Yorkshire and The Humber,41.67928107,2.801424804,551,50
4,2024,4,catt2,Person,Perception,England and Wales,,All people,45-54,gor,Yorkshire and The Humber,34.92154708,2.796132458,447,50


In [54]:
merge_df.isnull().sum()


Year                      0
Quarter                   0
MeasurementVar            0
MeasurementLevel          0
MeasurementType           0
Geography                 0
HouseholdType             0
Sex                       0
Age                       0
CharacteristicVar         0
Characteristic            0
Estimate                  0
StandardError             0
UnweightedCount           0
MeasurementVar_numeric    0
dtype: int64

In [55]:
# Check for duplicates  
duplicates = merge_df[merge_df.duplicated()]
duplicates_count = duplicates.shape[0]
print(f"Number of duplicate rows: {duplicates_count}")

Number of duplicate rows: 0


In [56]:
# Move measurement variable numeric to the front    
merge_df = merge_df[['MeasurementVar_numeric'] + [col for col in merge_df.columns if col != 'MeasurementVar_numeric']]

In [57]:
# remove unnecessary columns
merge_df = merge_df.drop(columns=['Quarter', 'MeasurementVar', 'MeasurementLevel', 'Geography', 'HouseholdType', 'Sex', 'CharacteristicVar', 'StandardError', 'UnweightedCount'])

In [58]:
print(merge_df)

     MeasurementVar_numeric  Year MeasurementType    Age  \
0                        50  2024      Perception  16-24   
1                        50  2024      Perception    16+   
2                        50  2024      Perception  25-34   
3                        50  2024      Perception  35-44   
4                        50  2024      Perception  45-54   
..                      ...   ...             ...    ...   
307                     100  2024      Perception  35-44   
308                     100  2024      Perception  45-54   
309                     100  2024      Perception  55-64   
310                     100  2024      Perception  65-74   
311                     100  2024      Perception    75+   

               Characteristic     Estimate  
0    Yorkshire and The Humber  51.84769986  
1    Yorkshire and The Humber  42.40532267  
2    Yorkshire and The Humber  43.83883414  
3    Yorkshire and The Humber  41.67928107  
4    Yorkshire and The Humber  34.92154708  
..       

In [59]:
# reference-measurementvar.csv by measurement label, sorted by measurementtype 'perception'
measurement_var_df = pd.read_csv('reference-measurementvar.csv')    
measurement_var_df = measurement_var_df[measurement_var_df['MeasurementType'] == 'Perception']
print(measurement_var_df)


    MeasurementVar                                   MeasurementLabel  \
0         abancar1  Perceived problem with abandoned or burnt out ...   
46        bvpiburg                             Worried about burglary   
47         bvpicar                            Worried about car crime   
48         bvpifrd                                Worried about fraud   
49        bvpiviol                        Worried about violent crime   
50           catt2  Police and local council are dealing with matt...   
51          catt2a  Police and local council are dealing with matt...   
52          catt2b  Police and local council are dealing with matt...   
53       cjsovb1dv                  Confident in effectiveness of CJS   
54        clcntry2  Perceived change in crime level in the past fe...   
55          clloc2  Perceived change in crime level in the past fe...   
66           drug1  Perceived problem with  people using or dealin...   
67          drunk1  Perceived problem with people b

In [60]:
# create updated csv 
merge_df.to_csv('perception_df.csv', index=False)